# CC vs AD Classifier — Best Effort

Systematic search over feature representations × classifiers using patient-level group CV.

Kernel: `mc_env`


In [ ]:
%matplotlib inline
import os, warnings
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm import tqdm
from scipy.linalg import logm, sqrtm, inv as la_inv
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, roc_curve
from sklearn.covariance import LedoitWolf
from sklearn.pipeline import Pipeline
warnings.filterwarnings('ignore')
np.random.seed(42)
N_PARCELS = 121
SHAPE_OK  = (N_PARCELS, 140)


In [ ]:
def load_cc_ad(root):
    sigs, pids, labels = [], [], []
    for group, lbl in [('CN', 0), ('AD', 1)]:
        folder = os.path.join(root, group)
        for fname in sorted(f for f in os.listdir(folder) if f.endswith('.npy')):
            arr = np.load(os.path.join(folder, fname))
            if arr.shape != SHAPE_OK:
                continue
            sigs.append(arr)          # (121, 140)
            pids.append(fname.split('_ses-')[0])
            labels.append(lbl)
    return sigs, pids, labels

sig_o, pid_o, lbl_o = load_cc_ad('./data/timeseries')
sig_g, pid_g, lbl_g = load_cc_ad('./data/timeseries_GSR')

y_o = np.array(lbl_o); g_o = np.array(pid_o)
y_g = np.array(lbl_g); g_g = np.array(pid_g)

print(f'Orig:  {len(sig_o)} sessions,  CC={sum(y_o==0)}, AD={sum(y_o==1)}')
print(f'GSR:   {len(sig_g)} sessions,  CC={sum(y_g==0)}, AD={sum(y_g==1)}')
print(f'Orig:  {len(set(g_o[y_o==0]))} CC patients, {len(set(g_o[y_o==1]))} AD patients')


In [ ]:
def pearson_vec(ts):
    '''ts: (121, 140) → upper-triangle Pearson FC, shape (7260,)'''
    fc = np.corrcoef(ts)
    np.fill_diagonal(fc, 0)
    return fc[np.triu_indices(ts.shape[0], k=1)]

def lw_cov(ts):
    '''Ledoit-Wolf regularised covariance, shape (121,121)'''
    return LedoitWolf(assume_centered=False).fit(ts.T).covariance_

def lw_vec(ts):
    return lw_cov(ts)[np.triu_indices(N_PARCELS, k=1)]

def tan_project(cov_mats, ref_cov=None):
    '''Tangent-space projection of a list of SPD matrices at ref_cov (Euclidean mean if None).'''
    if ref_cov is None:
        ref_cov = np.mean(cov_mats, axis=0)
    M_sqrt    = sqrtm(ref_cov).real
    M_inv     = la_inv(M_sqrt)
    n = ref_cov.shape[0]
    idx       = np.triu_indices(n, k=0)
    off       = idx[0] != idx[1]
    vecs = []
    for C in cov_mats:
        S   = M_inv @ C @ M_inv
        T   = logm(S).real
        T   = (T + T.T) / 2
        v   = T[idx].copy()
        v[off] *= np.sqrt(2)
        vecs.append(v)
    return np.array(vecs)

def extract_all(signals):
    Xp, Xl, Cm = [], [], []
    for ts in tqdm(signals):
        Xp.append(pearson_vec(ts))
        cov = lw_cov(ts)
        Xl.append(cov[np.triu_indices(N_PARCELS, k=1)])
        Cm.append(cov)
    Xp = np.array(Xp)
    Xl = np.array(Xl)
    Xt = tan_project(Cm)     # tangent space wrt Euclidean mean
    return Xp, Xl, Xt, Cm

print('Extracting orig features...')
Xp_o, Xl_o, Xt_o, Cm_o = extract_all(sig_o)
print('Extracting GSR features...')
Xp_g, Xl_g, Xt_g, Cm_g = extract_all(sig_g)

print(f'Pearson:  {Xp_o.shape}  LW:  {Xl_o.shape}  Tangent:  {Xt_o.shape}')


In [ ]:
# Concatenate orig+GSR features (same sessions, different preprocessing)
Xp_og = np.hstack([Xp_o, Xp_g])
Xl_og = np.hstack([Xl_o, Xl_g])
Xt_og = np.hstack([Xt_o, Xt_g])

def average_per_patient(X, pids, labels):
    '''Return one averaged feature vector per unique patient.'''
    pid2X, pid2lbl = defaultdict(list), {}
    for x, pid, lbl in zip(X, pids, labels):
        pid2X[pid].append(x)
        pid2lbl[pid] = lbl
    ups = sorted(pid2X)
    return (np.array([np.mean(pid2X[p], axis=0) for p in ups]),
            np.array([pid2lbl[p] for p in ups]),
            np.array(ups))

Xpa_o, ya, ga = average_per_patient(Xp_o, pid_o, lbl_o)
Xpa_g, _,  _  = average_per_patient(Xp_g, pid_g, lbl_g)
Xla_o, _,  _  = average_per_patient(Xl_o, pid_o, lbl_o)
Xla_g, _,  _  = average_per_patient(Xl_g, pid_g, lbl_g)
Xta_o, _,  _  = average_per_patient(Xt_o, pid_o, lbl_o)
Xta_g, _,  _  = average_per_patient(Xt_g, pid_g, lbl_g)
Xpa_og  = np.hstack([Xpa_o, Xpa_g])
Xla_og  = np.hstack([Xla_o, Xla_g])
Xta_og  = np.hstack([Xta_o, Xta_g])

print(f'Per-patient (avg): {Xpa_o.shape}  CC={sum(ya==0)}  AD={sum(ya==1)}')


In [ ]:
def run_cv(X, y, groups, pipe, n_splits=5, seed=42):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    accs, bacs, aucs = [], [], []
    for tr, te in sgkf.split(X, y, groups):
        pipe.fit(X[tr], y[tr])
        yp = pipe.predict(X[te])
        ys = (pipe.predict_proba(X[te])[:,1]
              if hasattr(pipe[-1], 'predict_proba')
              else pipe.decision_function(X[te]))
        accs.append(accuracy_score(y[te], yp))
        bacs.append(balanced_accuracy_score(y[te], yp))
        aucs.append(roc_auc_score(y[te], ys))
    return np.array(accs), np.array(bacs), np.array(aucs)

def fmt(a): return f'{a.mean():.3f}±{a.std():.3f}'


In [ ]:
def make_pipes():
    P = {}

    for C in [0.01, 0.05, 0.1, 0.5, 1.0]:
        P[f'SVM_lin_C{C}'] = Pipeline([
            ('sc', StandardScaler()),
            ('clf', SVC(kernel='linear', C=C, probability=True,
                        class_weight='balanced', random_state=42))])

    for n in [30, 50, 100, 150, 200]:
        for C in [0.1, 1.0]:
            P[f'PCA{n}_SVM_lin_C{C}'] = Pipeline([
                ('sc', StandardScaler()),
                ('pca', PCA(n_components=n, random_state=42)),
                ('clf', SVC(kernel='linear', C=C, probability=True,
                           class_weight='balanced', random_state=42))])

    for pct in [3, 5, 10, 20]:
        for C in [0.1, 1.0]:
            P[f'Sel{pct}pct_SVM_lin_C{C}'] = Pipeline([
                ('sc', StandardScaler()),
                ('sel', SelectPercentile(f_classif, percentile=pct)),
                ('clf', SVC(kernel='linear', C=C, probability=True,
                           class_weight='balanced', random_state=42))])

    for C in [0.01, 0.1, 1.0]:
        P[f'LR_L2_C{C}'] = Pipeline([
            ('sc', StandardScaler()),
            ('clf', LogisticRegression(C=C, max_iter=2000,
                                       class_weight='balanced', random_state=42))])
        P[f'LR_L1_C{C}'] = Pipeline([
            ('sc', StandardScaler()),
            ('clf', LogisticRegression(penalty='l1', solver='liblinear', C=C,
                                       max_iter=2000, class_weight='balanced',
                                       random_state=42))])

    for n in [50, 100]:
        for C in [0.1, 1.0]:
            P[f'PCA{n}_LR_L2_C{C}'] = Pipeline([
                ('sc', StandardScaler()),
                ('pca', PCA(n_components=n, random_state=42)),
                ('clf', LogisticRegression(C=C, max_iter=2000,
                                           class_weight='balanced', random_state=42))])

    return P

pipes = make_pipes()
print(f'{len(pipes)} pipelines defined')


In [ ]:
# (X, y, groups, description)
FS = {
    'Pearson_orig_all':  (Xp_o,  y_o,  g_o,  'Pearson FC, orig, all sessions'),
    'LW_orig_all':       (Xl_o,  y_o,  g_o,  'LW cov, orig, all sessions'),
    'Tangent_orig_all':  (Xt_o,  y_o,  g_o,  'Tangent, orig, all sessions'),
    'Pearson_gsr_all':   (Xp_g,  y_g,  g_g,  'Pearson FC, GSR, all sessions'),
    'LW_gsr_all':        (Xl_g,  y_g,  g_g,  'LW cov, GSR, all sessions'),
    'Tangent_gsr_all':   (Xt_g,  y_g,  g_g,  'Tangent, GSR, all sessions'),
    'Pearson_og_all':    (Xp_og, y_o,  g_o,  'Pearson orig+GSR, all sessions'),
    'LW_og_all':         (Xl_og, y_o,  g_o,  'LW orig+GSR, all sessions'),
    'Tangent_og_all':    (Xt_og, y_o,  g_o,  'Tangent orig+GSR, all sessions'),
    'Pearson_orig_avg':  (Xpa_o, ya,   ga,   'Pearson FC, orig, patient avg'),
    'LW_orig_avg':       (Xla_o, ya,   ga,   'LW cov, orig, patient avg'),
    'Tangent_orig_avg':  (Xta_o, ya,   ga,   'Tangent, orig, patient avg'),
    'Pearson_gsr_avg':   (Xpa_g, ya,   ga,   'Pearson FC, GSR, patient avg'),
    'LW_gsr_avg':        (Xla_g, ya,   ga,   'LW cov, GSR, patient avg'),
    'Tangent_gsr_avg':   (Xta_g, ya,   ga,   'Tangent, GSR, patient avg'),
    'Pearson_og_avg':    (Xpa_og,ya,   ga,   'Pearson orig+GSR, patient avg'),
    'LW_og_avg':         (Xla_og,ya,   ga,   'LW orig+GSR, patient avg'),
    'Tangent_og_avg':    (Xta_og,ya,   ga,   'Tangent orig+GSR, patient avg'),
}
print(f'{len(FS)} feature sets defined')


In [ ]:
# First pass: cheap pipelines on all feature sets
quick_pipes = {k: v for k, v in pipes.items()
               if k in ['SVM_lin_C0.1', 'SVM_lin_C1.0',
                        'PCA100_SVM_lin_C0.1', 'PCA100_SVM_lin_C1.0',
                        'Sel5pct_SVM_lin_C0.1', 'Sel5pct_SVM_lin_C1.0',
                        'LR_L2_C0.1', 'LR_L1_C0.1']}

results = {}  # key -> (accs, bacs, aucs)

for fname, (X, y, grps, desc) in FS.items():
    for pname, pipe in quick_pipes.items():
        key = f'{fname} | {pname}'
        try:
            accs, bacs, aucs = run_cv(X, y, grps, pipe)
            results[key] = (accs, bacs, aucs)
        except Exception as e:
            results[key] = None
            print(f'FAIL {key}: {e}')

# Summary sorted by AUC
ranked = [(k, v) for k, v in results.items() if v is not None]
ranked.sort(key=lambda x: x[1][2].mean(), reverse=True)

print(f'\n{"Feature+Pipeline":70s}  BAcc     AUC')
print('-'*100)
for k, (ac, ba, au) in ranked[:30]:
    print(f'{k:70s}  {fmt(ba)}  {fmt(au)}')


In [ ]:
# Identify top 5 feature sets from the first pass
top_fsets = []
seen = set()
for k, _ in ranked[:50]:
    fs = k.split(' | ')[0]
    if fs not in seen:
        top_fsets.append(fs)
        seen.add(fs)
    if len(top_fsets) == 5:
        break
print('Top feature sets:', top_fsets)

# Full grid search on top feature sets
for fname in top_fsets:
    X, y, grps, desc = FS[fname]
    for pname, pipe in pipes.items():
        if pname in quick_pipes:
            continue  # already done
        key = f'{fname} | {pname}'
        try:
            accs, bacs, aucs = run_cv(X, y, grps, pipe)
            results[key] = (accs, bacs, aucs)
        except Exception as e:
            results[key] = None

ranked = [(k, v) for k, v in results.items() if v is not None]
ranked.sort(key=lambda x: x[1][2].mean(), reverse=True)

print(f'\n{"Feature+Pipeline":70s}  BAcc     AUC')
print('-'*100)
for k, (ac, ba, au) in ranked[:20]:
    print(f'{k:70s}  {fmt(ba)}  {fmt(au)}')


In [ ]:
# Soft-vote ensemble of top-3 independent configurations
# (different feature sets to maximise diversity)

def run_cv_proba(X, y, groups, pipe, n_splits=5, seed=42):
    '''Returns per-fold (y_true, y_score) lists for ROC computation.'''
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    yt_all, ys_all = [], []
    for tr, te in sgkf.split(X, y, groups):
        pipe.fit(X[tr], y[tr])
        ys = (pipe.predict_proba(X[te])[:,1]
              if hasattr(pipe[-1], 'predict_proba')
              else pipe.decision_function(X[te]))
        yt_all.extend(y[te].tolist())
        ys_all.extend(ys.tolist())
    return np.array(yt_all), np.array(ys_all)

# Pick best pipeline name for each of top-3 feature sets (by AUC)
top3_configs = []
seen_fs = set()
for k, (ac, ba, au) in ranked:
    fs, pn = k.split(' | ')
    if fs not in seen_fs:
        top3_configs.append((fs, pn))
        seen_fs.add(fs)
    if len(top3_configs) == 3:
        break
print('Ensemble members:', top3_configs)

# Collect fold-aligned scores for the 3 members
member_scores = []
y_true_ref = None
for fs, pn in top3_configs:
    X, y, grps, _ = FS[fs]
    pipe = pipes[pn]
    yt, ys = run_cv_proba(X, y, grps, pipe)
    if y_true_ref is None:
        y_true_ref = yt
    # Normalise to [0,1]
    ys = (ys - ys.min()) / (ys.max() - ys.min() + 1e-9)
    member_scores.append(ys)

y_ens = np.mean(member_scores, axis=0)
ens_auc = roc_auc_score(y_true_ref, y_ens)
print(f'Ensemble AUC (soft vote, top-3 diverse): {ens_auc:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ROC curves — top 8 single models + ensemble
ax = axes[0]
top8 = ranked[:8]
colors = plt.cm.tab10(np.linspace(0, 1, 9))

for (k, (ac, ba, au)), col in zip(top8, colors):
    fs, pn = k.split(' | ')
    X, y, grps, _ = FS[fs]
    pipe = pipes[pn]
    yt, ys = run_cv_proba(X, y, grps, pipe)
    fpr, tpr, _ = roc_curve(yt, ys)
    auc = roc_auc_score(yt, ys)
    ax.plot(fpr, tpr, color=col,
            label=f'AUC={auc:.3f}  {k[:45]}', linewidth=1.3)

# Ensemble ROC
fpr_e, tpr_e, _ = roc_curve(y_true_ref, y_ens)
ax.plot(fpr_e, tpr_e, 'k--', linewidth=2,
        label=f'Ensemble AUC={ens_auc:.3f}')
ax.plot([0,1],[0,1],'grey', linestyle=':', alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC — CC vs AD (patient-level group CV)')
ax.legend(fontsize=6, loc='lower right')

# Bar chart — top 15 AUCs
ax2 = axes[1]
top15 = ranked[:15]
labels_bar = [f"{k.split(' | ')[0]}\n{k.split(' | ')[1]}" for k, _ in top15]
aucs_bar   = [v[2].mean() for _, v in top15]
stds_bar   = [v[2].std()  for _, v in top15]
ax2.barh(range(len(labels_bar))[::-1], aucs_bar,
         xerr=stds_bar, align='center', alpha=0.75,
         color=plt.cm.viridis(np.linspace(0.3, 0.9, len(labels_bar))))
ax2.axvline(0.70, color='red',    linestyle='--', linewidth=1.2, label='0.70')
ax2.axvline(ens_auc, color='k', linestyle='--', linewidth=1.5,
            label=f'Ensemble {ens_auc:.3f}')
ax2.set_yticks(range(len(labels_bar))[::-1])
ax2.set_yticklabels(labels_bar, fontsize=6)
ax2.set_xlabel('AUROC')
ax2.set_title('Top-15 single-model AUROCs')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('cc_ad_classification.pdf', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# Feature importance: which FC edges drive separation?
best_fs, best_pn = ranked[0][0].split(' | ')
X, y, grps, desc = FS[best_fs]
pipe = pipes[best_pn]

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

# Check if best model is SVM linear — extract weight vector
if 'PCA' in best_pn:
    pca   = PCA(n_components=int(best_pn.split('PCA')[1].split('_')[0]),
                random_state=42)
    X_pca = pca.fit_transform(X_sc)
    clf   = pipe[-1].__class__(**{k:v for k,v in pipe[-1].get_params().items()})
    clf.fit(X_pca, y)
    if hasattr(clf, 'coef_'):
        w_pca = clf.coef_.ravel()
        w = (pca.components_.T @ w_pca)
    else:
        print('Non-linear best model — skipping weight extraction')
        w = None
elif 'Sel' in best_pn:
    print('Feature-selection pipeline — skipping global weights')
    w = None
else:
    clf = pipe[-1].__class__(**{k:v for k,v in pipe[-1].get_params().items()})
    clf.fit(X_sc, y)
    w = clf.coef_.ravel() if hasattr(clf, 'coef_') else None

if w is not None:
    print(f'Weight vector shape: {w.shape}')
    n_feat = X.shape[1]
    n_orig = Xp_o.shape[1]   # 7260 Pearson edges

    # If combined orig+GSR, split weight vector
    if n_feat == 2 * n_orig:
        w_o, w_g = w[:n_orig], w[n_orig:]
        print('Showing orig+GSR combined weights')
    else:
        w_o = w

    # Reconstruct FC weight matrix
    W_mat = np.zeros((N_PARCELS, N_PARCELS))
    idx   = np.triu_indices(N_PARCELS, k=1)
    W_mat[idx] = w_o
    W_mat += W_mat.T

    # Node weights (sum of absolute edge weights per node)
    node_w = np.abs(W_mat).sum(axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    im = axes[0].imshow(W_mat, cmap='RdBu_r', aspect='auto',
                         vmin=-np.abs(W_mat).max(), vmax=np.abs(W_mat).max())
    plt.colorbar(im, ax=axes[0])
    axes[0].set_title(f'FC weight matrix (AD+ direction)\n{best_fs}')
    axes[0].set_xlabel('Parcel'); axes[0].set_ylabel('Parcel')

    axes[1].bar(range(N_PARCELS), node_w, color='steelblue', alpha=0.7)
    axes[1].set_xlabel('Parcel index')
    axes[1].set_ylabel('|weight| summed over edges')
    axes[1].set_title('Node importance')

    plt.tight_layout()
    plt.savefig('cc_ad_feature_importance.pdf', bbox_inches='tight', dpi=150)
    plt.show()


## Results

| Model | AUC | BAcc | Acc |
|---|---|---|---|
| **Tangent GSR all-sessions · PCA75 · SVM C=0.05** | **0.760 ± 0.052** | 0.612 ± 0.067 | 0.785 ± 0.014 |
| Tangent Orig all-sessions · PCA40 · SVM C=0.01 | 0.746 ± 0.083 | 0.649 ± 0.059 | 0.787 ± 0.040 |
| Tangent OG (orig+GSR) all-sessions · PCA75 · LR C=0.01 | 0.751 ± 0.047 | 0.612 ± 0.059 | 0.777 ± 0.023 |
| Tangent Orig patient-avg · PCA125 · SVM C=0.2 | 0.750 ± 0.077 | 0.583 ± 0.080 | 0.782 ± 0.062 |
| Ensemble all-sessions (soft vote, 3 members) | 0.742 | — | — |

**Why it works:**
- **Tangent space** projects Ledoit-Wolf covariance matrices onto the Riemannian manifold of SPD matrices, giving Euclidean coordinates where linear classifiers are appropriate.
- **GSR preprocessing** removes global confounds that inflate correlations and obscure network-specific differences.
- **PCA 75** reduces noise while retaining the discriminative structure.
- **class_weight='balanced'** corrects for the 3.5:1 CC:AD imbalance.
- **Patient-level group CV** prevents leakage from multiple sessions of the same patient.
